"Data Validation — Globus Mart Inventory Project:
Using Python (pandas) to validate the dataset — checked for missing values, duplicate transactions, and mismatched product IDs across sheets, and cleaned a currency-formatted column that wasn't calculating correctly."



In [1]:
import pandas as pd

In [2]:
#Task 1: cleaning the ₹ currency columns.
Retail = pd.read_excel("./Globus Mart Project.xlsx")

In [3]:
print(Retail.dtypes)
#dtypes means datatypes in pandas
#str- string/text
#int64- whole number
#float64- decimal number
#bool- true/false

Transaction_ID                  str
Date                 datetime64[us]
Store_Location                  str
SKU_ID                          str
Quantity_Sold                 int64
Unit_Retail_Price               str
category                        str
Total_revenue                 int64
dtype: object


In [4]:
Retail['Unit_Retail_Price'] = (
    Retail['Unit_Retail_Price']
    .astype(str) #astype means "as type" — it is used to change the data type of values in Pandas.
    .str.replace('₹', '', regex=False) #remove ₹ 
    .str.replace(',', '', regex=False) #remove , and regex=False tells Pandas to treat ₹ and , as ordinary characters
    .astype(float) #convert string into Float type
)

# Confirm it worked
print(Retail['Unit_Retail_Price'].dtype)   # should now say float64
print(Retail['Unit_Retail_Price'].head())

float64
0    175.0
1    250.0
2     20.0
3     15.0
4     45.0
Name: Unit_Retail_Price, dtype: float64


In [5]:
# Load the Inventory_Table sheet (separate from Retail, which is your Sales_Log)
Inventory = pd.read_excel("./Globus Mart Project.xlsx", sheet_name="Inventory_Table")

# Check every column for how many blank cells it has
print(Inventory.isna().sum()) #isna() checks whether each cell is blank/missing.

SKU_ID                        0
Product_Name                  0
Category                      0
Retail_Price                  0
Current_Stock                 0
Supplier_Name                 0
Total_Units_sold              0
Total_Revenue                 0
Avg_Units_per_transaction     0
Max_Units_per_Transaction     0
Cumulative_Revenue_percent    0
ABC_Class                     0
Jan_Sales                     0
Feb_sales                     0
March_Sales                   0
April_Sales                   0
May-Sales                     0
June_Sales                    0
StdDev_Monthly_sales          0
Average_Mothly_Sales          0
CV                            0
XYZ_Sales                     0
ABC_XYZ_Sales                 0
Avg_Lead_time_Days            0
Max_Lead_Time_Days            0
Safety_Stock                  0
ROP                           0
Stock_Status                  0
dtype: int64


In [6]:
# Check for exact duplicate rows in your Sales_Log
duplicate_count = Retail.duplicated().sum()
print("Number of duplicate rows:", duplicate_count)

# If there are any, see what they look like
if duplicate_count > 0:
    print(Retail[Retail.duplicated()])

Number of duplicate rows: 0


In [7]:
import numpy as np

# Step 1: convert monthly average demand into a daily rate
Inventory['Daily_Avg_Demand'] = Inventory['Average_Mothly_Sales'] / 30

# Step 2: convert monthly standard deviation into a daily standard deviation
Inventory['Daily_StdDev'] = Inventory['StdDev_Monthly_sales'] / np.sqrt(30)

# Step 3: Safety Stock using a 95% service level (Z = 1.65)
Inventory['Safety_Stock_Corrected'] = (
    1.65 * Inventory['Daily_StdDev'] * np.sqrt(Inventory['Avg_Lead_time_Days'])
).round(0)

# Step 4: Reorder Point = daily demand during lead time + safety buffer
Inventory['ROP_Corrected'] = (
    Inventory['Daily_Avg_Demand'] * Inventory['Avg_Lead_time_Days']
    + Inventory['Safety_Stock_Corrected']
).round(0)

# Compare old vs new side-by-side
print(Inventory[['Product_Name', 'Safety_Stock', 'Safety_Stock_Corrected', 'ROP', 'ROP_Corrected']])

              Product_Name  Safety_Stock  Safety_Stock_Corrected  ROP  \
0   Fortune Mustard Oil 1L    110.076923                    10.0  192   
1   Non-Stick Cookware Set     17.333333                     1.0   36   
2         Amul Butter 500g     35.571429                     3.0   60   
3       Premium Dinner Set      5.333333                     1.0   12   
4      Kitchen Storage Set     16.000000                     1.0   28   
5  Seasonal Decoration Set      6.000000                     0.0   10   
6          Coca-Cola 750ml    121.666667                     3.0  240   
7         Lays Classic 50g    220.000000                     5.0  392   
8   Maggi 2-Minute Noodles    264.000000                     3.0  500   

   ROP_Corrected  
0           16.0  
1            1.0  
2            4.0  
3            1.0  
4            1.0  
5            0.0  
6            7.0  
7           11.0  
8           11.0  


In [8]:
# Save both cleaned sheets back into a NEW Excel file
# (safer than overwriting your original — keeps a backup automatically)
with pd.ExcelWriter("Globus Mart Project_Cleaned.xlsx", engine='openpyxl') as writer:
    Retail.to_excel(writer, sheet_name='Sales_Log', index=False)
    Inventory.to_excel(writer, sheet_name='Inventory_Table', index=False)